# Step 8: HMM and Viterbi

Use the tuned ranker's per-point OOF probabilities to tune decoder settings. Reuse the same helpers for stacking once its held-out predictions arrive.

For speed, select **one complete training trip per saved fold**, closest to 700 GPS points. Selection uses trip sizes and folds only. This is a compact decoder-development sample, not an exhaustive search or an unbiased new CV estimate. No student-test label is used to choose settings.

Candidate misses remain wrong. Missing OOF emissions, including flagged groups, split the sequence; they are never filled using reference routes.

In [1]:
%run 08_hmm_helpers.ipynb
from datetime import datetime
from zoneinfo import ZoneInfo

model_folder = Path("results/step7_xgb/tuned_sample10000")
output = Path("results/hmm/ranker")
output.mkdir(parents=True, exist_ok=True)
contract = json.loads(Path("outputs/folds/contract.json").read_text())
summary = pd.read_csv("outputs/candidates/summary.csv", dtype={"trip_id": str})
folds = pd.read_csv("outputs/folds/trip_folds.csv", dtype={"trip_id": str})
summary = summary.merge(folds, on="trip_id", validate="one_to_one")
summary["size_difference"] = (summary["gps_points"] - 700).abs()
sample = summary.sort_values(["size_difference", "trip_id"]).groupby("fold").first().reset_index()
train_points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
train_points = train_points[train_points["trip_id"].isin(sample["trip_id"])].merge(
    pd.read_csv(contract["fold_file"], usecols=["group_id", "fold"]), on="group_id", validate="one_to_one")
ids = set(train_points["group_id"])
oof_parts = []
for chunk in pd.read_csv(model_folder / "oof_tuned_sample10000.csv", chunksize=250000):
    oof_parts.append(chunk[chunk["group_id"].isin(ids)])
oof = pd.concat(oof_parts, ignore_index=True)
assert (oof["fold"] == oof["group_id"].map(train_points.set_index("group_id")["fold"])).all()
sample.to_csv(output / "tuning_trips.csv", index=False)
print("Decoder tuning trips:", sample["trip_id"].tolist())

Decoder tuning trips: ['00000027', '00000037', '00000079', '00000045', '00000029']


## Tune on training OOF only

Try a 3×3 grid. Each trip is decoded independently, then correct selected segments are counted over its valid labelled points. Show per-fold metrics, then choose by pooled Top-1. Deterministic ties prefer smaller transition and emission weights.

In [2]:
if (output / "hmm_config.json").exists():
    config = json.loads((output / "hmm_config.json").read_text())
    assert config["tuning_source"] == str(model_folder / "oof_tuned_sample10000.csv")
    assert config["tuning_trips"] == sample["trip_id"].tolist()
    assert config["tuning_source_bytes"] == (model_folder / "oof_tuned_sample10000.csv").stat().st_size
    assert config["tuning_source_mtime_ns"] == (model_folder / "oof_tuned_sample10000.csv").stat().st_mtime_ns
    assert (output / "tuning_grid.csv").exists() and (output / "tuning_per_fold.csv").exists()
    print("Reusing frozen OOF-only decoder configuration:", config)
else:
    prepared = {}
    for row in sample.itertuples(index=False):
        point_rows = train_points[train_points["trip_id"] == row.trip_id]
        scores = oof[oof["group_id"].isin(point_rows["group_id"])]
        network = load_network(row.trip_id)
        prepared[row.trip_id] = prepare_trip(scores, point_rows, network)
        print("Prepared training trip:", row.trip_id, flush=True)
    
    records = []
    for beta in [0.25, 1.0, 4.0]:
        for emission_weight in [0.5, 1.0, 2.0]:
            for row in sample.itertuples(index=False):
                decoded, _ = decode_trip(prepared[row.trip_id], beta, emission_weight, posterior=False)
                valid = train_points[(train_points["trip_id"] == row.trip_id) & ~train_points["bad_gps"]]
                joined = valid.merge(decoded[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
                correct = joined["true_arc_id"].eq(joined["segment_id"]).sum()
                records.append({"beta": beta, "emission_weight": emission_weight, "trip_id": row.trip_id,
                    "fold": row.fold, "points": len(valid), "correct": correct, "top1": correct / len(valid)})
            print("Tuned beta", beta, "emission weight", emission_weight, flush=True)
    trials = pd.DataFrame(records)
    trials.to_csv(output / "tuning_per_fold.csv", index=False)
    results = trials.groupby(["beta", "emission_weight"], as_index=False)[["points", "correct"]].sum()
    results["top1"] = results["correct"] / results["points"]
    results.to_csv(output / "tuning_grid.csv", index=False)
    best = results.sort_values(["top1", "beta", "emission_weight"], ascending=[False, True, True]).iloc[0]
    config = {"beta": float(best["beta"]), "emission_weight": float(best["emission_weight"]),
        "temperature": 1.0, "gap_reset_s": 30, "transition_distance_scale_floor_m": 20, "unreachable_penalty": 20,
        "transition_cost_cap": 100, "topology": "directed reachability through full graph",
        "transition_distance": "endpoint-bridge lower-bound surrogate; not shortest road distance",
        "tuning_source": str(model_folder / "oof_tuned_sample10000.csv"),
        "tuning_source_bytes": (model_folder / "oof_tuned_sample10000.csv").stat().st_size,
        "tuning_source_mtime_ns": (model_folder / "oof_tuned_sample10000.csv").stat().st_mtime_ns,
        "tuning_trips": sample["trip_id"].tolist(), "tuning_folds": sample["fold"].tolist(),
        "tuning_points": int(best["points"]), "tuning_top1": float(best["top1"]),
        "posterior": "forward-backward marginals; Viterbi joint path reported separately",
        "selected_at": datetime.now(ZoneInfo("Asia/Kolkata")).isoformat()}
    (output / "hmm_config.json").write_text(json.dumps(config, indent=2) + "\n")
    display(results.sort_values("top1", ascending=False))
    del prepared


Reusing frozen OOF-only decoder configuration: {'beta': 1.0, 'emission_weight': 2.0, 'temperature': 1.0, 'gap_reset_s': 30, 'transition_distance_scale_floor_m': 20, 'unreachable_penalty': 20, 'transition_cost_cap': 100, 'topology': 'directed reachability through full graph', 'transition_distance': 'endpoint-bridge lower-bound surrogate; not shortest road distance', 'tuning_source': 'results/step7_xgb/tuned_sample10000/oof_tuned_sample10000.csv', 'tuning_source_bytes': 535555230, 'tuning_source_mtime_ns': 1794990600779689097, 'tuning_trips': ['00000027', '00000037', '00000079', '00000045', '00000029'], 'tuning_folds': [0, 1, 2, 3, 4], 'tuning_points': 3433, 'tuning_top1': 0.9297990096125838, 'posterior': 'forward-backward marginals; Viterbi joint path reported separately', 'selected_at': '2026-11-18T14:00:58.163659+05:30'}


In [3]:
valid = train_points[~train_points["bad_gps"]].copy()
ranker_scores = oof.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
winners = ranker_scores.loc[ranker_scores.groupby("group_id")["score"].idxmax(), ["group_id", "segment_id"]]
joined = valid.merge(winners, on="group_id", how="left", validate="one_to_one")
joined["correct"] = joined["true_arc_id"].eq(joined["segment_id"])
baseline = joined.groupby(["trip_id", "fold"], as_index=False).agg(points=("group_id", "size"), correct=("correct", "sum"))
baseline["top1"] = baseline["correct"] / baseline["points"]
baseline.to_csv(output / "tuning_baseline.csv", index=False)
ranker_top1 = float(joined["correct"].sum() / len(valid))
tuning_comparison = {"points": len(valid), "ranker_only_top1": ranker_top1,
    "hmm_top1": config["tuning_top1"],
    "gain_percentage_points": 100 * (config["tuning_top1"] - ranker_top1),
    "note": "Same five whole OOF tuning trips and valid-point denominator; development score, not independent test."}
(output / "tuning_comparison.json").write_text(json.dumps(tuning_comparison, indent=2) + "\n")
print("Same-sample decoder development comparison:", tuning_comparison)
del oof, ranker_scores

Same-sample decoder development comparison: {'points': 3433, 'ranker_only_top1': 0.9295077191960385, 'hmm_top1': 0.9297990096125838, 'gain_percentage_points': 0.029129041654529786, 'note': 'Same five whole OOF tuning trips and valid-point denominator; development score, not independent test.'}


## Decode student-test emissions

Read model probabilities for **all** observations, including flagged ones, then decode without inspecting their reference labels. Fill empty-shortlist observations with `segment_id = -1`. Quality labels are used only later by notebook 09.

Save both the Viterbi path and the candidate marginal posteriors. Decoder-only timing excludes feature generation and base-model prediction; final evaluation must add the latter.

In [3]:
scores = pd.read_csv(model_folder / "student_test_predictions.csv")
points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str},
    usecols=["trip_id", "group_id", "point_id", "time", "lon", "lat"])
selected_parts, probability_parts, timings = [], [], []
for trip_id, point_rows in points.groupby("trip_id", sort=False):
    start = perf_counter()
    network = load_network(trip_id, "student_test")
    emissions = scores[scores["group_id"].isin(point_rows["group_id"])]
    prepared = prepare_trip(emissions, point_rows, network, config["gap_reset_s"])
    decoded, posterior = decode_trip(prepared, config["beta"], config["emission_weight"])
    elapsed = perf_counter() - start
    complete = point_rows[["trip_id", "group_id", "point_id"]].merge(
        decoded[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
    complete["segment_id"] = complete["segment_id"].fillna(-1).astype("int64")
    selected_parts.append(complete)
    if posterior is not None:
        probability_parts.append(posterior)
    timings.append({"trip_id": trip_id, "points": len(point_rows), "decoder_s": elapsed})
    print("Decoded student-test trip:", trip_id, "seconds", round(elapsed, 2), flush=True)
selected = pd.concat(selected_parts, ignore_index=True)
probabilities = pd.concat(probability_parts, ignore_index=True).sort_values(["group_id", "segment_id"])
assert selected["group_id"].is_unique and len(selected) == len(points)
np.testing.assert_allclose(probabilities.groupby("group_id")["score"].sum(), 1, atol=1e-6)
selected.to_csv(output / "decoded_points.csv", index=False)
probabilities.to_parquet(output / "candidate_probabilities.parquet", index=False)
pd.DataFrame(timings).to_csv(output / "timings.csv", index=False)
(output / "timings.json").write_text(json.dumps({"decoder_predict_s": sum(r["decoder_s"] for r in timings),
    "timing_scope": "network loading/projection, directed topology, transition preparation, Viterbi and forward-backward",
    "includes": "network loading/projection, directed topology, transition preparation, Viterbi and forward-backward",
    "excludes": "feature generation and base model inference"}, indent=2) + "\n")
print("Saved Viterbi rows:", len(selected), "| Posterior rows:", len(probabilities))

Decoded student-test trip: 00000000 seconds 7.67


Decoded student-test trip: 00000007 seconds 3.7


Decoded student-test trip: 00000009 seconds 5.37


Decoded student-test trip: 00000040 seconds 8.23


Decoded student-test trip: 00000052 seconds 0.95


Decoded student-test trip: 00000056 seconds 18.85


Decoded student-test trip: 00000065 seconds 3.24


Decoded student-test trip: 00000076 seconds 6.39


Decoded student-test trip: 00000077 seconds 14.33


Decoded student-test trip: 00000091 seconds 3.33


Saved Viterbi rows: 19160 | Posterior rows: 1186138
